In [1]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data/raw")

train_file = data_dir / "UNSW_NB15_training-set.csv"
test_file = data_dir / "UNSW_NB15_testing-set.csv"

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

print("Training:", train_df.shape)
print("Testing:", test_df.shape)

Training: (175341, 45)
Testing: (82332, 45)


In [2]:
drop_cols = ["id", "attack_cat"]

X_train = train_df.drop(columns=drop_cols + ["label"])
y_train = train_df["label"]

X_test = test_df.drop(columns=drop_cols + ["label"])
y_test = test_df["label"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (175341, 42)
y_train: (175341,)
X_test: (82332, 42)
y_test: (82332,)


In [3]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ["proto", "service", "state"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        )
    ],
    remainder="passthrough"
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoded X_train:", X_train_encoded.shape)
print("Encoded X_test:", X_test_encoded.shape)

Encoded X_train: (175341, 194)
Encoded X_test: (82332, 194)


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print("Scaled X_train:", X_train_scaled.shape)
print("Scaled X_test:", X_test_scaled.shape)

Scaled X_train: (175341, 194)
Scaled X_test: (82332, 194)


In [5]:
import numpy as np

print("NaN in X_train_scaled:", np.isnan(X_train_scaled).sum())
print("NaN in X_test_scaled:", np.isnan(X_test_scaled).sum())

print("Inf in X_train_scaled:", np.isinf(X_train_scaled).sum())
print("Inf in X_test_scaled:", np.isinf(X_test_scaled).sum())

NaN in X_train_scaled: 0
NaN in X_test_scaled: 0
Inf in X_train_scaled: 0
Inf in X_test_scaled: 0


In [6]:
import joblib
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save NumPy arrays
np.save(processed_dir / "X_train_scaled.npy", X_train_scaled)
np.save(processed_dir / "X_test_scaled.npy", X_test_scaled)
np.save(processed_dir / "y_train.npy", y_train.to_numpy())
np.save(processed_dir / "y_test.npy", y_test.to_numpy())

# Save preprocessing objects
joblib.dump(preprocessor, processed_dir / "preprocessor.joblib")
joblib.dump(scaler, processed_dir / "scaler.joblib")

print("Processed files saved to:", processed_dir)

Processed files saved to: ..\data\processed


In [7]:
for file in processed_dir.iterdir():
    print(file.name)

preprocessor.joblib
scaler.joblib
X_test_scaled.npy
X_train_scaled.npy
y_test.npy
y_train.npy
